# Eval Data Prep

Build evaluation story pairs for embedding-model cosine similarity tests from:
- `tell_me_again_v1`
- `movie_remakes` (or fallback path `MovieRemakeDataset_NAACL2018`)

Target (configurable):
- 100 pairs from Tell Me Again
- 100 pairs from Movie Remakes
- In each dataset: 50% positive (true retell/remake), 50% random negative


In [1]:
import json
import random
from pathlib import Path
import pandas as pd

In [2]:
def find_project_root() -> Path:
    candidates = [Path.cwd().resolve(), Path.cwd().resolve().parent]
    for c in candidates:
        if (c / '.git').exists() and (c / 'data').exists() and (c / 'src').exists():
            return c
    raise RuntimeError('Could not locate project root from current working directory.')

PROJECT_ROOT = find_project_root()
DATA_DIR = PROJECT_ROOT / 'data'
print('PROJECT_ROOT:', PROJECT_ROOT)
print('DATA_DIR:', DATA_DIR)


PROJECT_ROOT: /Users/shayan/Projects/NarrativeSimilarity
DATA_DIR: /Users/shayan/Projects/NarrativeSimilarity/data


In [3]:
CONFIG = {
    'seed': 42,
    'tell_me_again_total_pairs': 100,
    'movie_remakes_total_pairs': 100,
    'positive_ratio': 0.5,

    # Paths
    'tell_me_again_dir': DATA_DIR / 'tell_me_again_v1',
    'movie_remakes_dir': DATA_DIR / 'MovieRemakeDataset_NAACL2018',

    # Output
    'output_json': DATA_DIR / 'eval_data' / 'eval_story_pairs_200.json',
    'output_csv': DATA_DIR / 'eval_data' / 'eval_story_pairs_200.csv',
}

random.seed(CONFIG['seed'])


print('tell_me_again_dir:', CONFIG['tell_me_again_dir'])
print('movie_remakes_dir:', CONFIG['movie_remakes_dir'])
print('output_json:', CONFIG['output_json'])
print('output_csv:', CONFIG['output_csv'])

tell_me_again_dir: /Users/shayan/Projects/NarrativeSimilarity/data/tell_me_again_v1
movie_remakes_dir: /Users/shayan/Projects/NarrativeSimilarity/data/MovieRemakeDataset_NAACL2018
output_json: /Users/shayan/Projects/NarrativeSimilarity/data/eval_data/eval_story_pairs_200.json
output_csv: /Users/shayan/Projects/NarrativeSimilarity/data/eval_data/eval_story_pairs_200.csv


## Explore Tell Me Again (dev split when available)


In [6]:
tma_dir = CONFIG['tell_me_again_dir']
dev_csv = tma_dir / 'dev_stories.csv'
summaries_root = tma_dir / 'summaries'

if not tma_dir.exists():
    raise FileNotFoundError(f'tell_me_again_v1 folder not found: {tma_dir}')

if not dev_csv.exists():
    raise FileNotFoundError(f'dev_stories.csv not found: {dev_csv}')

dev_ids = pd.read_csv(dev_csv, header=None, names=['wikidata_id'])
dev_ids['wikidata_id'] = dev_ids['wikidata_id'].astype(str).str.strip()

def load_tma_summary_json(wikidata_id: str):
    p = summaries_root / wikidata_id[:2] / f'{wikidata_id}.json'
    if not p.exists():
        return None
    try:
        with open(p, 'r', encoding='utf-8') as f:
            return json.load(f)
    except Exception:
        return None

def collect_tma_texts(obj: dict):
    texts = []

    # summaries is usually a dict: lang -> summary_text
    s = obj.get('summaries')
    if isinstance(s, dict):
        for lang, val in s.items():
            if isinstance(val, str) and val.strip():
                texts.append((f'summaries:{lang}', val.strip()))
    elif isinstance(s, list):
        for i, val in enumerate(s):
            if isinstance(val, str) and val.strip():
                texts.append((f'summaries:{i}', val.strip()))

    # en_translated_summaries is usually a dict: lang -> {'text': ...} or lang -> str
    ts = obj.get('en_translated_summaries')
    if isinstance(ts, dict):
        for lang, val in ts.items():
            if isinstance(val, dict):
                txt = val.get('text')
                if isinstance(txt, str) and txt.strip():
                    texts.append((f'en_translated_summaries:{lang}', txt.strip()))
            elif isinstance(val, str) and val.strip():
                texts.append((f'en_translated_summaries:{lang}', val.strip()))
    elif isinstance(ts, list):
        for i, val in enumerate(ts):
            if isinstance(val, str) and val.strip():
                texts.append((f'en_translated_summaries:{i}', val.strip()))
            elif isinstance(val, dict):
                txt = val.get('text')
                if isinstance(txt, str) and txt.strip():
                    texts.append((f'en_translated_summaries:{i}', txt.strip()))

    # de-duplicate by text while preserving order
    out = []
    seen = set()
    for source, txt in texts:
        key = txt.strip()
        if key in seen:
            continue
        seen.add(key)
        out.append((source, key))
    return out

records = []
for wid in dev_ids['wikidata_id'].tolist():
    obj = load_tma_summary_json(wid)
    if not isinstance(obj, dict):
        continue

    variants = collect_tma_texts(obj)
    if len(variants) < 2:
        continue

    for j, (src_name, txt) in enumerate(variants):
        records.append({
            'dataset': 'tell_me_again',
            'group_id': wid,
            'story_id': f'{wid}_{j}',
            'story_text': txt,
            'title': obj.get('title_en') or obj.get('title') or '',
            'source_variant': src_name,
        })

tma_stories_df = pd.DataFrame(records)
expected_cols = ['dataset', 'group_id', 'story_id', 'story_text', 'title', 'source_variant']
if tma_stories_df.empty:
    tma_stories_df = pd.DataFrame(columns=expected_cols)

print('dev_ids:', len(dev_ids))
print('usable story snippets:', len(tma_stories_df))
if tma_stories_df.empty:
    print('groups with >=2 retellings: 0 (no usable dev stories found)')
else:
    print('groups with >=2 retellings:', tma_stories_df.groupby('group_id').size().ge(2).sum())
    print('avg variants per group:', round(float(tma_stories_df.groupby('group_id').size().mean()), 2))
tma_stories_df.head(3)

dev_ids: 2950
usable story snippets: 19247
groups with >=2 retellings: 2949
avg variants per group: 6.53


,dataset,group_id,story_id,story_text,title,source_variant
0,tell_me_again,753610,753610_0,Ruby is a woman in her early 20s and the narra...,Ruby in Paradise,summaries:en
1,tell_me_again,753610,753610_1,Ruby quitte le Tennessee pour Panama City (Flo...,Ruby in Paradise,summaries:fr
2,tell_me_again,753610,753610_2,Ruby (Judd) es una joven que deja su pequeño p...,Ruby in Paradise,summaries:es


## Explore Movie Remakes (dev split if present, else fallback)


In [7]:
mr_dir = CONFIG['movie_remakes_dir']
instances_csv = mr_dir / 'testInstances.csv'
clean_tsv = mr_dir / 'movieRemakesManuallyCleaned.tsv'

if not mr_dir.exists():
    raise FileNotFoundError(f'movie remakes folder not found: {mr_dir}')
if not instances_csv.exists():
    raise FileNotFoundError(f'testInstances.csv not found: {instances_csv}')
if not clean_tsv.exists():
    raise FileNotFoundError(f'movieRemakesManuallyCleaned.tsv not found: {clean_tsv}')

inst = pd.read_csv(instances_csv)
inst.columns = [c.strip() for c in inst.columns]
# Expected columns after strip: clusterid, movieid
if 'clusterid' not in inst.columns or 'movieid' not in inst.columns:
    raise ValueError(f'Unexpected columns in {instances_csv}: {list(inst.columns)}')

inst['clusterid'] = inst['clusterid'].astype(str).str.strip()
inst['movieid'] = inst['movieid'].astype(str).str.strip()
valid_movie_ids = set(inst['movieid'])

rows = []
with clean_tsv.open('r', encoding='utf-8', errors='ignore') as f:
    for line in f:
        line = line.rstrip('\n')
        if not line:
            continue
        cols = line.split('\t')
        # Format: cluster_id, then repeated triplets (movieid, title, summary)
        if len(cols) < 4:
            continue
        cluster_id = cols[0].strip()
        for i in range(1, len(cols) - 2, 3):
            movie_id = cols[i].strip()
            title = cols[i + 1].strip()
            summary = cols[i + 2].strip()
            if not movie_id or not summary:
                continue
            # Keep only movie IDs in testInstances to match benchmark split source
            if movie_id not in valid_movie_ids:
                continue
            rows.append({
                'dataset': 'movie_remakes',
                'group_id': cluster_id,
                'story_id': movie_id,
                'story_text': summary,
                'title': title,
            })

mr_stories_df = pd.DataFrame(rows)
expected_cols = ['dataset', 'group_id', 'story_id', 'story_text', 'title']
if mr_stories_df.empty:
    mr_stories_df = pd.DataFrame(columns=expected_cols)
else:
    mr_stories_df = (
        mr_stories_df.drop_duplicates(subset=['story_id'])
        .assign(group_id=lambda d: d['group_id'].astype(str))
        .copy()
    )

print('instances rows:', len(inst))
print('usable clean summaries:', len(mr_stories_df))
if not mr_stories_df.empty and 'group_id' in mr_stories_df.columns:
    print('clusters with >=2 movies:', mr_stories_df.groupby('group_id').size().ge(2).sum())
else:
    print('clusters with >=2 movies: 0 (no usable stories found)')
mr_stories_df.head(3)


instances rows: 466
usable clean summaries: 466
clusters with >=2 movies: 181


,dataset,group_id,story_id,story_text,title
0,movie_remakes,1,14141235,The jury decides whether a young Chechen boy i...,12_(2007_film)
1,movie_remakes,1,11081144,After the final closing arguments have been pr...,12_Angry_Men_(1997_film)
2,movie_remakes,1,11094452,The story begins in a courtroom where a teenag...,Ek_Ruka_Hua_Faisla


## Pair Builders


In [6]:
def sample_positive_pairs(stories_df: pd.DataFrame, n_pairs: int, rng: random.Random):
    """
    Positive pairs: same group_id, different story_id.
    """
    candidates = []
    for gid, grp in stories_df.groupby('group_id'):
        ids = grp['story_id'].tolist()
        for i in range(len(ids)):
            for j in range(i + 1, len(ids)):
                a = grp.iloc[i]
                b = grp.iloc[j]
                candidates.append({
                    'dataset': a['dataset'],
                    'pair_type': 'positive',
                    'group_a': a['group_id'],
                    'group_b': b['group_id'],
                    'story_id_a': a['story_id'],
                    'story_id_b': b['story_id'],
                    'story_text_a': a['story_text'],
                    'story_text_b': b['story_text'],
                    'label': 1,
                })
    rng.shuffle(candidates)
    return candidates[:min(n_pairs, len(candidates))], len(candidates)


def sample_negative_pairs(stories_df: pd.DataFrame, n_pairs: int, rng: random.Random, candidates_df: pd.DataFrame | None = None):
    """
    Random negatives: different group_id.
    """
    base_df = candidates_df if candidates_df is not None else stories_df
    rows = base_df.to_dict(orient='records')
    if len(rows) < 2:
        return [], 0

    pair_set = set()
    out = []
    max_attempts = n_pairs * 60
    attempts = 0

    while len(out) < n_pairs and attempts < max_attempts:
        attempts += 1
        a, b = rng.sample(rows, 2)
        if a['group_id'] == b['group_id']:
            continue

        key = tuple(sorted([a['story_id'], b['story_id']]))
        if key in pair_set:
            continue
        pair_set.add(key)

        out.append({
            'dataset': a['dataset'],
            'pair_type': 'random_negative',
            'group_a': a['group_id'],
            'group_b': b['group_id'],
            'story_id_a': a['story_id'],
            'story_id_b': b['story_id'],
            'story_text_a': a['story_text'],
            'story_text_b': b['story_text'],
            'label': 0,
        })

    return out, len(out)


def build_eval_pairs(stories_df: pd.DataFrame, total_pairs: int, positive_ratio: float, seed: int, negative_candidates_df: pd.DataFrame | None = None):
    rng = random.Random(seed)
    n_pos = int(total_pairs * positive_ratio)
    n_neg = total_pairs - n_pos

    pos_pairs, pos_pool = sample_positive_pairs(stories_df, n_pos, rng)
    neg_pairs, neg_actual = sample_negative_pairs(stories_df, n_neg, rng, candidates_df=negative_candidates_df)

    built = pos_pairs + neg_pairs
    rng.shuffle(built)

    meta = {
        'requested_total': total_pairs,
        'requested_pos': n_pos,
        'requested_neg': n_neg,
        'built_total': len(built),
        'built_pos': sum(1 for x in built if x['label'] == 1),
        'built_neg': sum(1 for x in built if x['label'] == 0),
        'positive_pool_size': pos_pool,
        'negatives_built': neg_actual,
    }
    return built, meta

## Build 100 Tell-Me-Again + 100 Movie-Remakes Pairs


In [7]:
# Tell-Me-Again: restrict BOTH positives and negatives to English-text variants only.
# English-text variants are either raw English summaries or translated-to-English summaries.
tma_english_mask = tma_stories_df['source_variant'].astype(str).str.startswith('summaries:en') |                    tma_stories_df['source_variant'].astype(str).str.startswith('en_translated_summaries:')

tma_english_df = tma_stories_df[tma_english_mask].copy()

print('Tell Me Again total variants:', len(tma_stories_df))
print('Tell Me Again English variants for pairing:', len(tma_english_df))

# Use English-only pool for both positive and negative construction.
tma_pairs, tma_meta = build_eval_pairs(
    tma_english_df,
    total_pairs=CONFIG['tell_me_again_total_pairs'],
    positive_ratio=CONFIG['positive_ratio'],
    seed=CONFIG['seed'],
    negative_candidates_df=tma_english_df,
)

mr_pairs, mr_meta = build_eval_pairs(
    mr_stories_df,
    total_pairs=CONFIG['movie_remakes_total_pairs'],
    positive_ratio=CONFIG['positive_ratio'],
    seed=CONFIG['seed'] + 7,
)

print('Tell Me Again meta:', tma_meta)
print('Movie Remakes meta:', mr_meta)


Tell Me Again total variants: 19247
Tell Me Again English variants for pairing: 10936
Tell Me Again meta: {'requested_total': 100, 'requested_pos': 50, 'requested_neg': 50, 'built_total': 100, 'built_pos': 50, 'built_neg': 50, 'positive_pool_size': 16771, 'negatives_built': 50}
Movie Remakes meta: {'requested_total': 100, 'requested_pos': 50, 'requested_neg': 50, 'built_total': 100, 'built_pos': 50, 'built_neg': 50, 'positive_pool_size': 251, 'negatives_built': 50}


## Combine, Inspect, and Save


In [8]:
all_pairs = tma_pairs + mr_pairs
for i, p in enumerate(all_pairs):
    p['pair_id'] = f"{p['dataset']}__{i:04d}"

pairs_df = pd.DataFrame(all_pairs)

print('Total pairs:', len(pairs_df))
print('By dataset:')
print(pairs_df['dataset'].value_counts().to_string())
print('By dataset x pair_type:')
print(pairs_df.groupby(['dataset', 'pair_type']).size().to_string())
pairs_df[['pair_id', 'dataset', 'pair_type', 'story_id_a', 'story_id_b', 'label']].head(10)

Total pairs: 200
By dataset:
dataset
tell_me_again    100
movie_remakes    100
By dataset x pair_type:
dataset        pair_type      
movie_remakes  positive           50
               random_negative    50
tell_me_again  positive           50
               random_negative    50


,pair_id,dataset,pair_type,story_id_a,story_id_b,label
0,tell_me_again__0000,tell_me_again,random_negative,174284_5,461447_6,0
1,tell_me_again__0001,tell_me_again,random_negative,111018582_4,11618_6,0
2,tell_me_again__0002,tell_me_again,random_negative,323401_0,1398816_0,0
3,tell_me_again__0003,tell_me_again,positive,1627168_0,1627168_4,1
4,tell_me_again__0004,tell_me_again,positive,1536329_7,1536329_8,1
5,tell_me_again__0005,tell_me_again,random_negative,159808_0,1759361_0,0
6,tell_me_again__0006,tell_me_again,random_negative,183063_5,4004171_4,0
7,tell_me_again__0007,tell_me_again,random_negative,1593107_0,1382263_4,0
8,tell_me_again__0008,tell_me_again,positive,1168478_2,1168478_3,1
9,tell_me_again__0009,tell_me_again,random_negative,18420573_6,1621611_0,0


In [9]:
CONFIG['output_json'].parent.mkdir(parents=True, exist_ok=True)

# JSON is best for long texts; CSV is also saved for convenience.
pairs_df.to_json(CONFIG['output_json'], orient='records', indent=2, force_ascii=False)
pairs_df.to_csv(CONFIG['output_csv'], index=False)

print('Saved JSON:', CONFIG['output_json'])
print('Saved CSV :', CONFIG['output_csv'])

Saved JSON: /Users/shayan/Projects/NarrativeSimilarity/data/eval_data/eval_story_pairs_200.json
Saved CSV : /Users/shayan/Projects/NarrativeSimilarity/data/eval_data/eval_story_pairs_200.csv


## Notes

- `label=1` means true retelling/remake pair (positive).
- `label=0` means random cross-group pair (negative).
- To change counts later, edit `tell_me_again_total_pairs` and `movie_remakes_total_pairs` in `CONFIG`.
- For cosine eval later, you can embed `story_text_a` and `story_text_b` and compute similarity per `pair_id`.


In [12]:
# Quick sanity check: load eval_data CSV and print head/basic stats.
from pathlib import Path

csv_path = DATA_DIR / 'eval_data' / 'eval_story_pairs_200.csv'
if not csv_path.exists():
    raise FileNotFoundError(f'CSV not found at: {csv_path}')

eval_df = pd.read_csv(csv_path)

print('Loaded:', csv_path)
print('Shape:', eval_df.shape)
print('\nHead:')
print(eval_df.head().to_string(index=False))

print('\nBasic stats:')
if 'dataset' in eval_df.columns:
    print('By dataset:')
    print(eval_df['dataset'].value_counts(dropna=False).to_string())

if 'pair_type' in eval_df.columns:
    print('\nBy pair_type:')
    print(eval_df['pair_type'].value_counts(dropna=False).to_string())

if {'dataset', 'pair_type'}.issubset(eval_df.columns):
    print('\nBy dataset x pair_type:')
    print(eval_df.groupby(['dataset', 'pair_type']).size().to_string())

if 'label' in eval_df.columns:
    print('\nLabel distribution:')
    print(eval_df['label'].value_counts(dropna=False).to_string())

for col in ['story_text_a', 'story_text_b']:
    if col in eval_df.columns:
        lengths = eval_df[col].fillna('').astype(str).str.len()
        print(f"\n{col} length stats:")
        print(lengths.describe().to_string())



Loaded: /Users/shayan/Projects/NarrativeSimilarity/data/eval_data/eval_story_pairs_200.csv
Shape: (200, 10)

Head:
      dataset       pair_type   group_a  group_b  story_id_a story_id_b                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                              

In [13]:
eval_df.head(20)

,dataset,pair_type,group_a,group_b,story_id_a,story_id_b,story_text_a,story_text_b,label,pair_id
0,tell_me_again,random_negative,174284,461447,174284_5,461447_6,"In 1936, archaeology professor and adventurer ...",Amelia Earhart was an American aviation pionee...,0,tell_me_again__0000
1,tell_me_again,random_negative,111018582,11618,111018582_4,11618_6,"Young Alfred ""Al"" Yankovic is interested in pa...","# General presentation\nIn an industrial town,...",0,tell_me_again__0001
2,tell_me_again,random_negative,323401,1398816,323401_0,1398816_0,"In 1874, the U.S. government encroaches on the...",Mark Dixon is a police detective who was just ...,0,tell_me_again__0002
3,tell_me_again,positive,1627168,1627168,1627168_0,1627168_4,The books switches from character to character...,"The story opens on the morning of Sunday, Octo...",1,tell_me_again__0003
4,tell_me_again,positive,1536329,1536329,1536329_7,1536329_8,"Roy Clayton, an FBI agent, and his colleague M...",Samir Horn (Don Cheadle) is an American ex-sol...,1,tell_me_again__0004
5,tell_me_again,random_negative,159808,1759361,159808_0,1759361_0,"Matko Destanov, a small-time Romani smuggler a...",Sonny Hooper (Burt Reynolds) is the stunt coor...,0,tell_me_again__0005
6,tell_me_again,random_negative,183063,4004171,183063_5,4004171_4,# Content\nSuccessful child psychiatrist Dr. M...,The two friends Folcacchio and Gulfardo are on...,0,tell_me_again__0006
7,tell_me_again,random_negative,1593107,1382263,1593107_0,1382263_4,"The film takes place in the summer of 1941, af...",A prominent New England doctor has been murder...,0,tell_me_again__0007
8,tell_me_again,positive,1168478,1168478,1168478_2,1168478_3,It is set in Jamaica in the mid-19th century a...,"Antoniette, a young landowner in Jamaica in th...",1,tell_me_again__0008
9,tell_me_again,random_negative,18420573,1621611,18420573_6,1621611_0,# Hang up the phone\nAt the beginning of the 2...,As Trevor (Jason Connery) drifts through Texas...,0,tell_me_again__0009


## Retrieval-Style Evaluation Set

Another useful evaluation format is **retrieval evaluation**:
- each row is a **query story**,
- there are **5 candidate options**,
- exactly **1 option is positive** (same narrative group as the query),
- the other **4 options are negatives** (different groups).

This can evaluate whether an embedding model ranks the true matching story above distractors.

In this setup:
- we build **100 datapoints per dataset** (`tell_me_again`, `movie_remakes`),
- Tell-Me-Again datapoints are constrained to **English text only** (including translated-to-English variants),
- positives for Tell-Me-Again are selected from that English-only pool.


In [8]:
# Build retrieval-style eval dataframe: 1 query + 5 options (1 positive, 4 negatives).
# Saves both CSV and JSON.

import random

RETRIEVAL_PER_DATASET = 100
RETRIEVAL_OPTIONS = 5
NEG_PER_QUERY = RETRIEVAL_OPTIONS - 1
RETRIEVAL_SEED = CONFIG['seed'] + 101

rng = random.Random(RETRIEVAL_SEED)


def build_retrieval_eval_for_dataset(df: pd.DataFrame, dataset_name: str, n_queries: int, rng: random.Random):
    if df.empty:
        return pd.DataFrame(), {'dataset': dataset_name, 'requested': n_queries, 'built': 0, 'reason': 'empty dataframe'}

    by_group = {gid: g.to_dict('records') for gid, g in df.groupby('group_id')}

    eligible_queries = []
    for gid, rows in by_group.items():
        if len(rows) >= 2:
            eligible_queries.extend(rows)

    if not eligible_queries:
        return pd.DataFrame(), {'dataset': dataset_name, 'requested': n_queries, 'built': 0, 'reason': 'no groups with >=2 stories'}

    all_rows = df.to_dict('records')

    rng.shuffle(eligible_queries)

    results = []
    used_queries = set()
    attempts = 0
    max_attempts = n_queries * 200

    while len(results) < n_queries and attempts < max_attempts:
        attempts += 1

        q = rng.choice(eligible_queries)
        qid = str(q['story_id'])
        qgid = q['group_id']

        if qid in used_queries and len(used_queries) < len(eligible_queries):
            continue

        pos_cands = [r for r in by_group[qgid] if str(r['story_id']) != qid]
        if not pos_cands:
            continue
        pos = rng.choice(pos_cands)

        neg_cands = [r for r in all_rows if r['group_id'] != qgid]
        if len(neg_cands) < NEG_PER_QUERY:
            continue
        negs = rng.sample(neg_cands, NEG_PER_QUERY)

        options = [
            {
                'option_story_id': str(pos['story_id']),
                'option_group_id': str(pos['group_id']),
                'option_text': pos['story_text'],
                'is_positive': 1,
            }
        ] + [
            {
                'option_story_id': str(n['story_id']),
                'option_group_id': str(n['group_id']),
                'option_text': n['story_text'],
                'is_positive': 0,
            }
            for n in negs
        ]

        rng.shuffle(options)

        correct_idx = next(i for i, opt in enumerate(options) if opt['is_positive'] == 1)

        row = {
            'dataset': dataset_name,
            'query_story_id': str(q['story_id']),
            'query_group_id': str(q['group_id']),
            'query_text': q['story_text'],
            'num_options': RETRIEVAL_OPTIONS,
            'correct_option_index': correct_idx,
        }

        for i, opt in enumerate(options):
            row[f'option_{i}_story_id'] = opt['option_story_id']
            row[f'option_{i}_group_id'] = opt['option_group_id']
            row[f'option_{i}_text'] = opt['option_text']
            row[f'option_{i}_is_positive'] = int(opt['is_positive'])

        results.append(row)
        used_queries.add(qid)

    out_df = pd.DataFrame(results)
    meta = {
        'dataset': dataset_name,
        'requested': n_queries,
        'built': len(out_df),
        'eligible_query_stories': len(eligible_queries),
        'unique_queries_used': len(used_queries),
        'attempts': attempts,
    }
    return out_df, meta


# Tell-Me-Again: force English-only pool for BOTH positives and negatives in retrieval setup.
tma_english_mask = tma_stories_df['source_variant'].astype(str).str.startswith('summaries:en') | tma_stories_df['source_variant'].astype(str).str.startswith('en_translated_summaries:')
tma_retrieval_pool = tma_stories_df[tma_english_mask].copy()

mr_retrieval_pool = mr_stories_df.copy()

tma_retrieval_df, tma_retrieval_meta = build_retrieval_eval_for_dataset(
    tma_retrieval_pool,
    dataset_name='tell_me_again',
    n_queries=RETRIEVAL_PER_DATASET,
    rng=rng,
)

mr_retrieval_df, mr_retrieval_meta = build_retrieval_eval_for_dataset(
    mr_retrieval_pool,
    dataset_name='movie_remakes',
    n_queries=RETRIEVAL_PER_DATASET,
    rng=rng,
)

retrieval_eval_df = pd.concat([tma_retrieval_df, mr_retrieval_df], ignore_index=True)

print('Tell Me Again retrieval meta:', tma_retrieval_meta)
print('Movie Remakes retrieval meta:', mr_retrieval_meta)
print('Total retrieval rows:', len(retrieval_eval_df))
print('By dataset:')
print(retrieval_eval_df['dataset'].value_counts().to_string())

out_dir = DATA_DIR / 'eval_data'
out_dir.mkdir(parents=True, exist_ok=True)
retrieval_csv = out_dir / 'retrieval_eval_df.csv'
retrieval_json = out_dir / 'retrieval_eval_df.json'

retrieval_eval_df.to_csv(retrieval_csv, index=False)
retrieval_eval_df.to_json(retrieval_json, orient='records', force_ascii=False, indent=2)

print('Saved retrieval CSV :', retrieval_csv)
print('Saved retrieval JSON:', retrieval_json)

retrieval_eval_df.head(3)


Tell Me Again retrieval meta: {'dataset': 'tell_me_again', 'requested': 100, 'built': 100, 'eligible_query_stories': 10933, 'unique_queries_used': 100, 'attempts': 101}
Movie Remakes retrieval meta: {'dataset': 'movie_remakes', 'requested': 100, 'built': 100, 'eligible_query_stories': 389, 'unique_queries_used': 100, 'attempts': 114}
Total retrieval rows: 200
By dataset:
dataset
tell_me_again    100
movie_remakes    100
Saved retrieval CSV : /Users/shayan/Projects/NarrativeSimilarity/data/eval_data/retrieval_eval_df.csv
Saved retrieval JSON: /Users/shayan/Projects/NarrativeSimilarity/data/eval_data/retrieval_eval_df.json


,dataset,query_story_id,query_group_id,query_text,num_options,correct_option_index,option_0_story_id,option_0_group_id,option_0_text,option_0_is_positive,...,option_2_text,option_2_is_positive,option_3_story_id,option_3_group_id,option_3_text,option_3_is_positive,option_4_story_id,option_4_group_id,option_4_text,option_4_is_positive
0,tell_me_again,1145932_0,1145932,"The novel opens in early 1945. Peter Marlowe, ...",5,1,1592308_0,1592308,"Roger Brown (Aksel Hennie), Norway's most succ...",0,...,"On July 2, 1937, Amelia Earhart (Hilary Swank)...",0,195402_0,195402,Notorious womanizer Michael James (Peter O' To...,0,2064383_6,2064383,"Three friends, in their senior year of college...",0
1,tell_me_again,1809883_8,1809883,End of the 17th century. A proud nobleman refu...,5,2,6530195_3,6530195,The film is set in a small town with the ficti...,0,...,"In England in the late 17th century, King Jame...",1,1592308_6,1592308,Roger Brown is leading a double life. Norway's...,0,27703213_0,27703213,"When tragedy strikes three families, their des...",0
2,tell_me_again,1996269_6,1996269,Surgeon Eugene Ferguson is held hostage by a g...,5,0,1996269_5,1996269,"Dr. Ferguson and his wife, Helen, were on vaca...",1,...,Pather Panchali is primarily a depiction of li...,0,2133691_6,2133691,"During the Mexican-American War, Captain John ...",0,862956_4,862956,"Cold War, late '40s, early '50s. A Soviet defe...",0


## Same-Theme (Event Re-order) Structural Similarity Labels

To evaluate the trained embedding model, we also need **continuous structural similarity scores** for story pairs that share the same theme but have different event orders. 
Because the structural similarity model expects event alignments, the pipeline is: 
(1) build event-like sentence lists for each story pair, 
(2) get LLM alignments between those event lists and save them, then 
(3) run `StructuralSimilarityModel` to compute final structural similarity scores.


In [9]:
# Build same-theme pairs, request LLM alignments, and save alignment records.
import json
import re
import time
from pathlib import Path

import pandas as pd
from tqdm.auto import tqdm
from openai import OpenAI

SYNTHETIC_PATH_CANDIDATES = [
    PROJECT_ROOT / "data" / "synthetic_stories.json",
    PROJECT_ROOT / "data" / "synthetic_data.json",
    PROJECT_ROOT / "synthetic_data.json",
]
ALIGN_OUT = PROJECT_ROOT / "data" / "eval_results" / "same_theme_story_pair_alignments.json"
ALIGN_OUT.parent.mkdir(parents=True, exist_ok=True)

SYNTHETIC_PATH = next((p for p in SYNTHETIC_PATH_CANDIDATES if p.exists()), None)
if SYNTHETIC_PATH is None:
    cand = "\n".join(str(p) for p in SYNTHETIC_PATH_CANDIDATES)
    raise FileNotFoundError(f"synthetic_data.json not found. Checked:\n{cand}")

raw_data = json.loads(SYNTHETIC_PATH.read_text(encoding="utf-8"))

# Support synthetic_stories.json format:
# [
#   {
#     "theme": "...",
#     "core_events": [...],
#     "story_1": "...",
#     "story_2": "...",
#     "story_3": "..."
#   },
#   ...
# ]
if isinstance(raw_data, list) and raw_data and isinstance(raw_data[0], dict) and {'story_1', 'story_2', 'story_3'}.issubset(raw_data[0].keys()):
    rows = []
    for ti, rec in enumerate(raw_data):
        theme = str(rec.get('theme', f'theme_{ti:03d}'))
        stories = [
            ('story_1', rec.get('story_1', '')),
            ('story_2', rec.get('story_2', '')),
            ('story_3', rec.get('story_3', '')),
        ]
        stories = [(sid, txt) for sid, txt in stories if isinstance(txt, str) and txt.strip()]

        # all within-theme pairs
        for i in range(len(stories)):
            for j in range(i + 1, len(stories)):
                sid_a, txt_a = stories[i]
                sid_b, txt_b = stories[j]
                rows.append({
                    'dataset': 'synthetic',
                    'pair_type': 'same_theme_positive',
                    'group_a': theme,
                    'group_b': theme,
                    'theme_a': theme,
                    'theme_b': theme,
                    'story_id_a': f'{theme}__{sid_a}',
                    'story_id_b': f'{theme}__{sid_b}',
                    'story_text_a': txt_a,
                    'story_text_b': txt_b,
                    'pair_id': f'synthetic__{ti:03d}__{sid_a}__{sid_b}',
                })

    pairs_df = pd.DataFrame(rows)

else:
    # Generic fallback formats: list of pair dicts or dict with pair lists.
    if isinstance(raw_data, dict):
        for k in ["pairs", "data", "records", "items"]:
            if isinstance(raw_data.get(k), list):
                pair_records = raw_data[k]
                break
        else:
            pair_records = [raw_data]
    elif isinstance(raw_data, list):
        pair_records = raw_data
    else:
        raise ValueError("synthetic source should be a list or dict containing records.")

    pairs_df = pd.DataFrame(pair_records)

    required_cols = ["story_text_a", "story_text_b"]
    missing = [c for c in required_cols if c not in pairs_df.columns]
    if missing:
        raise ValueError(f"synthetic source missing required story columns: {missing}")

    if "pair_id" not in pairs_df.columns:
        pairs_df["pair_id"] = [f"synthetic_pair_{i:06d}" for i in range(len(pairs_df))]

    # Ensure metadata columns exist for downstream save schema.
    for col, default in [
        ("dataset", "synthetic"),
        ("pair_type", "synthetic"),
        ("group_a", ""),
        ("group_b", ""),
        ("story_id_a", ""),
        ("story_id_b", ""),
    ]:
        if col not in pairs_df.columns:
            pairs_df[col] = default

# Keep only same-theme pairs when theme columns exist.
if "theme_a" in pairs_df.columns and "theme_b" in pairs_df.columns:
    same_theme_df = pairs_df[pairs_df["theme_a"].astype(str) == pairs_df["theme_b"].astype(str)].copy()
elif "theme" in pairs_df.columns:
    same_theme_df = pairs_df.copy()
elif "group_a" in pairs_df.columns and "group_b" in pairs_df.columns:
    same_theme_df = pairs_df[pairs_df["group_a"].astype(str) == pairs_df["group_b"].astype(str)].copy()
else:
    same_theme_df = pairs_df.copy()

if same_theme_df.empty:
    raise ValueError("No same-theme pairs found in synthetic source based on available theme/group columns.")

print(f"Loaded synthetic source: {SYNTHETIC_PATH}")
print(f"Total synthetic pairs: {len(pairs_df)} | same-theme pairs used: {len(same_theme_df)}")

def split_story_to_events(text: str):
    if not isinstance(text, str):
        return []
    chunks = re.split(r"(?<=[.!?])\s+", text.strip())
    return [c.strip() for c in chunks if c and c.strip()]

system_prompt = """You are a careful narrative event alignment assistant.

Your task is to identify structural correspondences between two ordered sequences of eventful sentences extracted from full stories.

Two sentences should be matched if they play a similar conceptual or thematic role in their respective sequence of sentences, even if the level of detail differs.

Important:
- One story may describe an episode using multiple detailed events, while the other may summarize it briefly.
- You may match one event to multiple events.
- You may match multiple events to one event.
- You may match groups of events to groups of events.
- Only match events if they represent the same underlying narrative episode or structural role.
- Be conservative and avoid vague matches.

Return STRICT JSON with this format:

{
  "matches": [
    {
      "a_indices": [<int>, ...],
      "b_indices": [<int>, ...]
    }
  ],
  "unmatched_a": [<int>, ...],
  "unmatched_b": [<int>, ...]
}

Indices are 1-based.
Do not include explanations.
Output JSON only.
"""

def make_user_prompt(events_a, events_b):
    a_lines = [f"{i+1}. {s}" for i, s in enumerate(events_a)]
    b_lines = [f"{i+1}. {s}" for i, s in enumerate(events_b)]
    return (
        "Story A events:\n" + "\n".join(a_lines) + "\n\n"
        + "Story B events:\n" + "\n".join(b_lines)
    )

key_path = PROJECT_ROOT / "openai_key.txt"
if not key_path.exists():
    key_path = PROJECT_ROOT.parent / "openai_key.txt"
if not key_path.exists():
    raise FileNotFoundError(f"openai_key.txt not found under {PROJECT_ROOT} or parent")
api_key = key_path.read_text(encoding="utf-8").strip()
if not api_key:
    raise ValueError(f"openai_key.txt is empty: {key_path}")

client = OpenAI(api_key=api_key)
MODEL_NAME = "gpt-5"
REQUEST_SLEEP_SECONDS = 0.2

existing = {}
if ALIGN_OUT.exists():
    raw = json.loads(ALIGN_OUT.read_text(encoding="utf-8"))
    if isinstance(raw, list):
        existing = {str(x["pair_id"]): x for x in raw if isinstance(x, dict) and "pair_id" in x}

results = list(existing.values())
seen = set(existing.keys())
pending_df = same_theme_df[~same_theme_df["pair_id"].astype(str).isin(seen)].copy()

def call_alignment_llm(events_a, events_b):
    user_prompt = make_user_prompt(events_a, events_b)
    resp = client.responses.create(
        model=MODEL_NAME,
        input=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
    )
    txt = getattr(resp, "output_text", "")
    if not txt:
        txt = json.dumps(resp.model_dump(), ensure_ascii=False)
    return txt

errors = 0
pbar = tqdm(pending_df.itertuples(index=False), total=len(pending_df), desc="Same-theme alignments", unit="pair")
for row in pbar:
    pair_id = str(row.pair_id)
    events_a = split_story_to_events(str(row.story_text_a))
    events_b = split_story_to_events(str(row.story_text_b))

    rec = {
        "pair_id": pair_id,
        "dataset": row.dataset,
        "pair_type": row.pair_type,
        "group_a": str(row.group_a),
        "group_b": str(row.group_b),
        "story_id_a": str(row.story_id_a),
        "story_id_b": str(row.story_id_b),
        "EventsA_align": events_a,
        "EventsB_align": events_b,
    }

    try:
        raw_text = call_alignment_llm(events_a, events_b).strip()
        alignment = json.loads(raw_text)
        rec["alignment"] = alignment
        rec["llm_model"] = MODEL_NAME
    except Exception as e:
        errors += 1
        rec["alignment"] = {"matches": [], "unmatched_a": [], "unmatched_b": []}
        rec["error"] = str(e)

    results.append(rec)
    seen.add(pair_id)
    ALIGN_OUT.write_text(json.dumps(results, ensure_ascii=False, indent=2), encoding="utf-8")
    pbar.set_postfix(total_saved=len(results), errors=errors)
    time.sleep(REQUEST_SLEEP_SECONDS)

print(f"Saved alignment records: {len(results)}")
print(f"Alignment file: {ALIGN_OUT}")
print(f"Errors: {errors}")


Loaded synthetic source: /tank/scratch/shayan/Projects/NarrativeSimilarity/data/synthetic_stories.json
Total synthetic pairs: 60 | same-theme pairs used: 60


Same-theme alignments: 100%|██████████| 60/60 [17:49<00:00, 17.82s/pair, errors=0, total_saved=60]

Saved alignment records: 60
Alignment file: /tank/scratch/shayan/Projects/NarrativeSimilarity/data/eval_results/same_theme_story_pair_alignments.json
Errors: 0


In [6]:
# Load saved alignments and compute final structural similarity scores with StructuralSimilarityModel.
import json
from pathlib import Path

import pandas as pd
from tqdm.auto import tqdm

from structural_similarity_model import StructuralSimilarityModel

ALIGN_OUT = PROJECT_ROOT / "data" / "eval_results" / "same_theme_story_pair_alignments.json"
SCORES_OUT = PROJECT_ROOT / "data" / "eval_results" / "same_theme_story_pair_structural_scores.json"

if not ALIGN_OUT.exists():
    raise FileNotFoundError(f"Alignment file not found: {ALIGN_OUT}")

alignment_rows = json.loads(ALIGN_OUT.read_text(encoding="utf-8"))
if not isinstance(alignment_rows, list) or len(alignment_rows) == 0:
    raise ValueError("Alignment file is empty or invalid.")

model = StructuralSimilarityModel()

scored_rows = []
for row in tqdm(alignment_rows, desc="Structural scoring", unit="pair"):
    base = {
        "pair_id": row.get("pair_id"),
        "dataset": row.get("dataset"),
        "pair_type": row.get("pair_type"),
        "group_a": row.get("group_a"),
        "group_b": row.get("group_b"),
        "story_id_a": row.get("story_id_a"),
        "story_id_b": row.get("story_id_b"),
    }

    try:
        pred = model.predict_similarity(row)
        scored_rows.append({**base, **pred})
    except Exception as e:
        scored_rows.append({**base, "error": str(e)})

SCORES_OUT.write_text(json.dumps(scored_rows, ensure_ascii=False, indent=2), encoding="utf-8")
scores_df = pd.DataFrame(scored_rows)
print("Saved:", SCORES_OUT)
print("Rows:", len(scores_df))
display_cols = [c for c in ["pair_id", "alignment_similarity", "semantic_similarity", "pred_event_rating_mean_joint", "error"] if c in scores_df.columns]
display(scores_df[display_cols].head(10))


Structural scoring: 100%|██████████| 60/60 [00:00<00:00, 73.94pair/s]

Saved: /tank/scratch/shayan/Projects/NarrativeSimilarity/data/eval_results/same_theme_story_pair_structural_scores.json
Rows: 60


,pair_id,alignment_similarity,semantic_similarity,pred_event_rating_mean_joint
0,synthetic__000__story_1__story_2,0.833333,0.709149,2.413691
1,synthetic__000__story_1__story_3,0.888889,0.715456,2.430809
2,synthetic__000__story_2__story_3,0.888889,0.755447,2.468156
3,synthetic__001__story_1__story_2,0.833333,0.802492,2.500864
4,synthetic__001__story_1__story_3,0.888889,0.774051,2.485530
5,synthetic__001__story_2__story_3,0.777778,0.745285,2.436210
6,synthetic__002__story_1__story_2,1.000000,0.744585,2.480468
7,synthetic__002__story_1__story_3,0.888889,0.776961,2.488248
8,synthetic__002__story_2__story_3,0.916667,0.708145,2.429595
9,synthetic__003__story_1__story_2,0.833333,0.693645,2.399212


In [9]:
# Build theme-level anchor eval file from same-theme structural scores.
import json
from pathlib import Path
import pandas as pd

synth_data_version = "v1"

SYNTHETIC_SRC = PROJECT_ROOT / 'data' / f"synthetic_stories_{synth_data_version}.json"
SCORES_SRC = PROJECT_ROOT / 'data' / 'eval_results' / 'same_theme_story_pair_structural_scores.json'
OUT_JSON = PROJECT_ROOT / 'data' / 'eval_data' / f'synthetic_{synth_data_version}_anchor_eval_data.json'
OUT_CSV = PROJECT_ROOT / 'data' / 'eval_data' / f'synthetic_{synth_data_version}_anchor_eval_data.csv'

if not SYNTHETIC_SRC.exists():
    raise FileNotFoundError(f"Missing synthetic source: {SYNTHETIC_SRC}")
if not SCORES_SRC.exists():
    raise FileNotFoundError(f"Missing structural scores: {SCORES_SRC}")

synthetic_obj = json.loads(SYNTHETIC_SRC.read_text(encoding="utf-8"))
themes = synthetic_obj["data"]
score_rows = json.loads(SCORES_SRC.read_text(encoding="utf-8"))

# map pair_id -> model score
pair_score = {str(r.get('pair_id')): r.get('pred_event_rating_mean_joint') for r in score_rows}

rows = []
for ti, rec in enumerate(themes):
    theme = str(rec.get('theme', f'theme_{ti:03d}'))
    s1 = str(rec.get('story_1', '')).strip()
    s2 = str(rec.get('story_2', '')).strip()
    s3 = str(rec.get('story_3', '')).strip()
    llm_label = rec.get("most_similar_to_story_1", float("nan"))
    if not (s1 and s2 and s3):
        continue

    pid_12 = f'synthetic__{ti:03d}__story_1__story_2'
    pid_13 = f'synthetic__{ti:03d}__story_1__story_3'
    pid_23 = f'synthetic__{ti:03d}__story_2__story_3'

    score_12 = pair_score.get(pid_12)
    score_13 = pair_score.get(pid_13)
    score_23 = pair_score.get(pid_23)

    if score_12 is None or score_13 is None:
        # anchor eval needs both anchor comparisons
        continue

    if score_12 > score_13:
        gt_higher = 'story_2'
    elif score_13 > score_12:
        gt_higher = 'story_3'
    else:
        gt_higher = 'tie'

    rows.append({
        'theme_index': ti,
        'theme': theme,
        'story_1': s1,
        'story_2': s2,
        'story_3': s3,
        'pair_id_12': pid_12,
        'pair_id_13': pid_13,
        'pair_id_23': pid_23,
        'struct_score_12': float(score_12),
        'struct_score_13': float(score_13),
        'struct_score_23': float(score_23) if score_23 is not None else None,
        'gt_higher_vs_story_1': gt_higher,
        'LLM_label': llm_label,
    })

eval_anchor_df = pd.DataFrame(rows)
OUT_JSON.parent.mkdir(parents=True, exist_ok=True)
OUT_JSON.write_text(json.dumps(rows, ensure_ascii=False, indent=2), encoding="utf-8")
eval_anchor_df.to_csv(OUT_CSV, index=False)

print("Saved:", OUT_JSON)
print("Saved:", OUT_CSV)
print("Rows:", len(eval_anchor_df))
if 'gt_higher_vs_story_1' in eval_anchor_df.columns:
    print(eval_anchor_df['gt_higher_vs_story_1'].value_counts(dropna=False))
display(eval_anchor_df.head(10))

Saved: /tank/scratch/shayan/Projects/NarrativeSimilarity/data/eval_data/synthetic_v1_anchor_eval_data.json
Saved: /tank/scratch/shayan/Projects/NarrativeSimilarity/data/eval_data/synthetic_v1_anchor_eval_data.csv
Rows: 20
gt_higher_vs_story_1
story_3    16
story_2     4
Name: count, dtype: int64


,theme_index,theme,story_1,story_2,story_3,pair_id_12,pair_id_13,pair_id_23,struct_score_12,struct_score_13,struct_score_23,gt_higher_vs_story_1,LLM_label
0,0,Starting a new job,After relocating to Chicago for a new position...,Maya first succeeded on a small assignment aft...,The interview convinced Maya to relocate acros...,synthetic__000__story_1__story_2,synthetic__000__story_1__story_3,synthetic__000__story_2__story_3,2.413691,2.430809,2.468156,story_3,NaN
1,1,Recovering after a breakup,"After separating from her partner, Nina isolat...",Nina began recovering only after reconnecting ...,The separation forced Nina to reflect on patte...,synthetic__001__story_1__story_2,synthetic__001__story_1__story_3,synthetic__001__story_2__story_3,2.500864,2.485530,2.436210,story_2,NaN
2,2,Training for a marathon,Carlos spent months training for his first mar...,"After nearly giving up because of an injury, C...",Carlos dreamed of finishing a marathon ever si...,synthetic__002__story_1__story_2,synthetic__002__story_1__story_3,synthetic__002__story_2__story_3,2.480468,2.488248,2.429595,story_3,NaN
3,3,Immigrating to a new country,"After traveling to Canada with his family, Ami...",Amir first felt a sense of belonging after mon...,Traveling abroad gave Amir opportunities he ne...,synthetic__003__story_1__story_2,synthetic__003__story_1__story_3,synthetic__003__story_2__story_3,2.399212,2.490237,2.403680,story_3,NaN
4,4,Launching a small business,Lena carefully planned her online bakery befor...,The business only began to grow after Lena cha...,"After months of planning, Lena invested everyt...",synthetic__004__story_1__story_2,synthetic__004__story_1__story_3,synthetic__004__story_2__story_3,2.358588,2.467124,2.390750,story_3,NaN
5,5,Caring for an aging parent,Daniel began visiting his mother every weekend...,The hardest part for Daniel was accepting how ...,Although Daniel initially believed occasional ...,synthetic__005__story_1__story_2,synthetic__005__story_1__story_3,synthetic__005__story_2__story_3,2.411314,2.546768,2.503384,story_3,NaN
6,6,Navigating high school friendships,Emma met a group of classmates during her firs...,After confronting her friends about being excl...,Emma first reconciled with her friends near th...,synthetic__006__story_1__story_2,synthetic__006__story_1__story_3,synthetic__006__story_2__story_3,2.464038,2.464799,2.407968,story_3,NaN
7,7,Surviving financial hardship,"After losing his job unexpectedly, Marcus borr...",Marcus struggled to keep up with rent after lo...,The period after Marcus lost his job forced hi...,synthetic__007__story_1__story_2,synthetic__007__story_1__story_3,synthetic__007__story_2__story_3,2.507807,2.504027,2.508252,story_2,NaN
8,8,Preparing for a major exam,After enrolling in a difficult certification c...,Priya only began studying seriously after a mo...,The pressure of completing the exam grew stron...,synthetic__008__story_1__story_2,synthetic__008__story_1__story_3,synthetic__008__story_2__story_3,2.511330,2.578381,2.446333,story_3,NaN
9,9,Healing after an accident,"After a serious bike crash, Olivia depended he...",Olivia struggled emotionally after the crash b...,The accident forced Olivia to step away from h...,synthetic__009__story_1__story_2,synthetic__009__story_1__story_3,synthetic__009__story_2__story_3,2.518242,2.526967,2.511499,story_3,NaN


## Build Ranking Evaluation Data from Retrieval Data (LLM-Labeled)

This section converts the retrieval-style evaluation set into a ranking task.
For each query story (anchor) and its 5 candidate stories, we ask `gpt-5.4` to rank all candidates from most structurally similar to least structurally similar.

Quality check: after each LLM response, we validate that rank-1 matches the known positive candidate from retrieval data (`correct_option_index`). If it does not match, we drop that datapoint and print its index so filtering is transparent.

The final ranking dataset is saved as both CSV and JSON under `data/eval_data/`.


In [4]:
# Create ranking task labels from retrieval_eval_df using gpt-5.4.
import json
import re
import time
from pathlib import Path

import pandas as pd
from openai import OpenAI
from tqdm.auto import tqdm

# Resolve project root robustly (not /src)
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'src':
    PROJECT_ROOT = PROJECT_ROOT.parent

RETRIEVAL_PATH = PROJECT_ROOT / 'data' / 'eval_data' / 'retrieval_eval_df.csv'
KEY_PATH = PROJECT_ROOT / 'openai_key.txt'
OUT_JSON = PROJECT_ROOT / 'data' / 'eval_data' / 'ranking_eval_df.json'
OUT_CSV = PROJECT_ROOT / 'data' / 'eval_data' / 'ranking_eval_df.csv'

if not RETRIEVAL_PATH.exists():
    raise FileNotFoundError(f'Retrieval eval file not found: {RETRIEVAL_PATH}')
if not KEY_PATH.exists():
    raise FileNotFoundError(f'OpenAI key file not found: {KEY_PATH}')

api_key = KEY_PATH.read_text(encoding='utf-8').strip()
if not api_key:
    raise ValueError(f'OpenAI key file is empty: {KEY_PATH}')

client = OpenAI(api_key=api_key)
MODEL_NAME = 'gpt-5.4'
REQUEST_SLEEP_SECONDS = 0.1

retrieval_df = pd.read_csv(RETRIEVAL_PATH)
required_cols = ['query_text', 'correct_option_index'] + [f'option_{i}_text' for i in range(5)]
missing = [c for c in required_cols if c not in retrieval_df.columns]
if missing:
    raise ValueError(f'Missing required retrieval columns: {missing}')

if 'pair_id' not in retrieval_df.columns:
    retrieval_df['pair_id'] = [f'retrieval_{i:05d}' for i in range(len(retrieval_df))]

system_prompt = (
    'You are a careful narrative structure evaluator. '
    'Given one anchor story and five candidate stories, rank the candidates by structural similarity '
    '(event progression, role of events, and narrative arc), from most similar (rank 1) to least similar (rank 5). '
    'Return JSON only.'
)


def _parse_ranking(raw_text: str):
    # Extract first JSON object from response
    m = re.search(r'\{.*\}', raw_text, flags=re.S)
    if not m:
        raise ValueError('No JSON object found in model response.')
    obj = json.loads(m.group(0))

    # Expected format:
    # {"ranking": [{"option_index": 0, "rank": 2}, ...]}
    ranking = obj.get('ranking')
    if not isinstance(ranking, list) or len(ranking) != 5:
        raise ValueError('Invalid ranking format: expected 5 ranking entries.')

    rank_by_option = {}
    used_ranks = set()
    for item in ranking:
        if not isinstance(item, dict):
            raise ValueError('Ranking entries must be objects.')
        oi = int(item.get('option_index'))
        rr = int(item.get('rank'))
        if oi < 0 or oi > 4:
            raise ValueError(f'option_index out of range: {oi}')
        if rr < 1 or rr > 5:
            raise ValueError(f'rank out of range: {rr}')
        if oi in rank_by_option:
            raise ValueError(f'duplicate option_index: {oi}')
        if rr in used_ranks:
            raise ValueError(f'duplicate rank: {rr}')
        rank_by_option[oi] = rr
        used_ranks.add(rr)

    if len(rank_by_option) != 5 or used_ranks != {1, 2, 3, 4, 5}:
        raise ValueError('Ranking must be a permutation over options 0..4 and ranks 1..5.')

    top_option = min(rank_by_option, key=lambda k: rank_by_option[k])
    ordered = [k for k, _ in sorted(rank_by_option.items(), key=lambda kv: kv[1])]
    return rank_by_option, top_option, ordered


def score_ranking(anchor: str, options: list[str]):
    user_prompt = (
        'Anchor story:'
        f'{anchor}'
        'Candidate stories:'
        f'option_0: {options[0]}'
        f'option_1:{options[1]}'
        f'option_2:{options[2]}'
        f'option_3:{options[3]}'
        f'option_4:{options[4]}'
        'Return STRICT JSON with this exact schema:'
        '{'
        '  "ranking": ['
        '    {"option_index": 0, "rank": 1},'
        '    {"option_index": 1, "rank": 2},'
        '    {"option_index": 2, "rank": 3},'
        '    {"option_index": 3, "rank": 4},'
        '    {"option_index": 4, "rank": 5}'
        '  ]'
        '}'
        'No explanations. JSON only.'
    )

    resp = client.responses.create(
        model=MODEL_NAME,
        input=[
            {'role': 'system', 'content': system_prompt},
            {'role': 'user', 'content': user_prompt},
        ],
    )

    raw_text = getattr(resp, 'output_text', '') or ''
    if not raw_text:
        raw_text = str(resp)

    rank_by_option, top_option, ordered = _parse_ranking(raw_text)
    return {
        'rank_by_option': rank_by_option,
        'top_option': top_option,
        'ordered_options': ordered,
        'raw_response': raw_text,
    }


rows_out = []
dropped = 0
errors = 0

pbar = tqdm(retrieval_df.itertuples(index=True), total=len(retrieval_df), desc='LLM ranking labels', unit='pair')
for row in pbar:
    idx = int(row.Index)
    try:
        anchor = str(row.query_text)
        options = [str(getattr(row, f'option_{i}_text')) for i in range(5)]
        gold = int(row.correct_option_index)

        out = score_ranking(anchor, options)

        # Validation gate: rank-1 must match retrieval positive.
        if int(out['top_option']) != gold:
            dropped += 1
            print(f'Dropping datapoint index {idx}: rank1={out["top_option"]} but gold={gold}')
            pbar.set_postfix(kept=len(rows_out), dropped=dropped, errors=errors)
            time.sleep(REQUEST_SLEEP_SECONDS)
            continue

        rec = {
            'row_index': idx,
            'pair_id': str(row.pair_id),
            'dataset': str(getattr(row, 'dataset', '')),
            'query_text': anchor,
            'correct_option_index': gold,
            'rank_1_option_index': int(out['ordered_options'][0]),
            'rank_2_option_index': int(out['ordered_options'][1]),
            'rank_3_option_index': int(out['ordered_options'][2]),
            'rank_4_option_index': int(out['ordered_options'][3]),
            'rank_5_option_index': int(out['ordered_options'][4]),
            'option_0_rank': int(out['rank_by_option'][0]),
            'option_1_rank': int(out['rank_by_option'][1]),
            'option_2_rank': int(out['rank_by_option'][2]),
            'option_3_rank': int(out['rank_by_option'][3]),
            'option_4_rank': int(out['rank_by_option'][4]),
            'llm_model': MODEL_NAME,
            'option_0_text': options[0],
            'option_1_text': options[1],
            'option_2_text': options[2],
            'option_3_text': options[3],
            'option_4_text': options[4],
            'raw_response': out['raw_response'],
        }
        rows_out.append(rec)

    except Exception as e:
        errors += 1
        print(f'Error at index {idx}: {e}')

    pbar.set_postfix(kept=len(rows_out), dropped=dropped, errors=errors)
    time.sleep(REQUEST_SLEEP_SECONDS)

out_df = pd.DataFrame(rows_out)
OUT_JSON.write_text(json.dumps(rows_out, ensure_ascii=False, indent=2), encoding='utf-8')
out_df.to_csv(OUT_CSV, index=False)

print('Done.')
print('Input rows        :', len(retrieval_df))
print('Kept ranking rows :', len(out_df))
print('Dropped (rank1!=gold):', dropped)
print('Errors            :', errors)
print('Saved JSON        :', OUT_JSON)
print('Saved CSV         :', OUT_CSV)
out_df.head(3)


LLM ranking labels:   0%|          | 0/200 [00:00<?, ?pair/s]

Dropping datapoint index 159: rank1=0 but gold=4
Done.
Input rows        : 200
Kept ranking rows : 199
Dropped (rank1!=gold): 1
Errors            : 0
Saved JSON        : /Users/shayan/Projects/NarrativeSimilarity/data/eval_data/ranking_eval_df.json
Saved CSV         : /Users/shayan/Projects/NarrativeSimilarity/data/eval_data/ranking_eval_df.csv


,row_index,pair_id,dataset,query_text,correct_option_index,rank_1_option_index,rank_2_option_index,rank_3_option_index,rank_4_option_index,rank_5_option_index,...,option_2_rank,option_3_rank,option_4_rank,llm_model,option_0_text,option_1_text,option_2_text,option_3_text,option_4_text,raw_response
0,0,retrieval_00000,tell_me_again,"The novel opens in early 1945. Peter Marlowe, ...",1,1,0,2,3,4,...,3,4,5,gpt-5.4,"Roger Brown (Aksel Hennie), Norway's most succ...","Set during World War II, the novel describes t...","On July 2, 1937, Amelia Earhart (Hilary Swank)...",Notorious womanizer Michael James (Peter O' To...,"Three friends, in their senior year of college...","{\n ""ranking"": [\n {""option_index"": 1, ""ra..."
1,1,retrieval_00001,tell_me_again,End of the 17th century. A proud nobleman refu...,2,2,0,3,1,4,...,1,3,5,gpt-5.4,The film is set in a small town with the ficti...,It's 1977. After a difficult childhood and ado...,"In England in the late 17th century, King Jame...",Roger Brown is leading a double life. Norway's...,"When tragedy strikes three families, their des...","{\n ""ranking"": [\n {""option_index"": 2, ""ra..."
2,2,retrieval_00002,tell_me_again,Surgeon Eugene Ferguson is held hostage by a g...,0,0,4,3,2,1,...,4,3,2,gpt-5.4,"Dr. Ferguson and his wife, Helen, were on vaca...","In 1951, Marcus left his native New Jersey to ...",Pather Panchali is primarily a depiction of li...,"During the Mexican-American War, Captain John ...","Cold War, late '40s, early '50s. A Soviet defe...","{\n ""ranking"": [\n {\n ""option_index""..."
